# ASL clips → fixed **60 frames** (match SGSL length)

**SGSL** clips in this project are typically **60 frames** (e.g. 30 fps × 2 s). This notebook **resamples every ASL `*.mp4`** so each output has exactly **60** frames for consistent model inputs.

**Method:** uniformly spaced indices over the original timeline (`numpy.linspace`). If a clip already has 60 frames, it is copied unchanged (same pixels / fps).

**Duration:** output **fps** is set to `fps_orig × (n_original / 60)` so wall‑clock duration matches the source clip.

**Encoding:** Writes MP4 via **`ffmpeg` + libx264** (piped raw BGR). OpenCV’s `mp4v` writer fails on many derived FPS values (MPEG‑4 timebase limit); **install ffmpeg** (“ffmpeg” on PATH, e.g. `brew install ffmpeg`).

```bash
pip install "opencv-python-headless>=4.8" "numpy>=1.26,<2"
```

```bash
# macOS — needed for exporting MP4 from this notebook
brew install ffmpeg
```

In [1]:
# Optional
# %pip install -q "opencv-python-headless>=4.8" "numpy>=1.26,<2"

In [2]:
from __future__ import annotations

import shutil
import subprocess
from pathlib import Path

import cv2
import numpy as np

# --- paths ---
PROJECT_ROOT = Path(".").resolve()
ASL_SRC = PROJECT_ROOT / "Videos" / "ASL"
ASL_DST = PROJECT_ROOT / "Videos" / "ASL_60"
SGSL_ROOT = PROJECT_ROOT / "Videos" / "SGSL"

TARGET_FRAMES = 60

In [3]:
def list_numeric_mp4s(folder: Path) -> list[Path]:
    paths = sorted(folder.glob("*.mp4"), key=lambda p: int(p.stem))
    return paths


def read_all_frames(path: Path) -> tuple[list[np.ndarray], float]:
    cap = cv2.VideoCapture(str(path))
    if not cap.isOpened():
        raise RuntimeError(f"Cannot open video: {path}")
    fps = float(cap.get(cv2.CAP_PROP_FPS))
    if fps <= 1e-3:
        fps = 30.0
    frames: list[np.ndarray] = []
    while True:
        ok, fr = cap.read()
        if not ok or fr is None:
            break
        frames.append(fr)
    cap.release()
    if not frames:
        raise RuntimeError(f"No frames read: {path}")
    return frames, fps


def write_mp4(path: Path, frames: list[np.ndarray], fps: float) -> None:
    """Encode H.264 MP4 via ffmpeg stdin (avoids OpenCV mp4v / MPEG‑4 timebase errors)."""
    path.parent.mkdir(parents=True, exist_ok=True)
    if not frames:
        raise ValueError("no frames")
    h, w = frames[0].shape[:2]
    fps_cmd = float(fps)
    if fps_cmd <= 1e-6:
        fps_cmd = 30.0

    ff = shutil.which("ffmpeg") or "ffmpeg"
    proc = subprocess.Popen(
        [
            ff,
            "-y",
            "-f",
            "rawvideo",
            "-vcodec",
            "rawvideo",
            "-pix_fmt",
            "bgr24",
            "-s",
            f"{w}x{h}",
            "-r",
            str(fps_cmd),
            "-i",
            "-",
            "-an",
            "-c:v",
            "libx264",
            "-pix_fmt",
            "yuv420p",
            "-preset",
            "veryfast",
            "-crf",
            "20",
            "-movflags",
            "+faststart",
            str(path),
        ],
        stdin=subprocess.PIPE,
        stdout=subprocess.DEVNULL,
        stderr=subprocess.PIPE,
    )
    assert proc.stdin is not None
    try:
        for fr in frames:
            if fr.shape[0] != h or fr.shape[1] != w:
                fr = cv2.resize(fr, (w, h), interpolation=cv2.INTER_LINEAR)
            if fr.dtype != np.uint8:
                fr = fr.astype(np.uint8)
            if not fr.flags["C_CONTIGUOUS"]:
                fr = np.ascontiguousarray(fr)
            proc.stdin.write(fr.tobytes())
    finally:
        proc.stdin.close()

    stderr = proc.stderr.read().decode("utf-8", errors="replace") if proc.stderr else ""
    if proc.wait() != 0:
        raise RuntimeError(
            "ffmpeg failed for " + str(path) + ". Install ffmpeg (e.g. brew install ffmpeg).\n"
            "--- stderr tail ---\n"
            + stderr[-4000:]
        )


def uniform_resample_frames(frames: list[np.ndarray], target: int) -> list[np.ndarray]:
    """Pick `target` frames by uniform indices along the clip (handles n < target via repeats)."""
    n = len(frames)
    if n == target:
        return frames
    idx = np.linspace(0, n - 1, target)
    idx = np.round(idx).astype(int)
    idx = np.clip(idx, 0, n - 1)
    return [frames[i] for i in idx]

In [4]:
def normalize_clip_to_target_frames(src: Path, dst: Path, target: int) -> dict:
    frames, fps = read_all_frames(src)
    n0 = len(frames)
    out_fps = fps * (n0 / target)
    out_frames = uniform_resample_frames(frames, target)
    assert len(out_frames) == target
    write_mp4(dst, out_frames, out_fps)
    try:
        src_rel = str(src.relative_to(PROJECT_ROOT))
    except ValueError:
        src_rel = str(src)
    return {"src": src_rel, "n_in": n0, "fps_in": fps, "fps_out": out_fps}


assert ASL_SRC.is_dir(), f"Missing folder: {ASL_SRC}"

# Fresh output tree each run (avoid mixing old clips). Comment out to incremental update.
if ASL_DST.is_dir():
    shutil.rmtree(ASL_DST)
ASL_DST.mkdir(parents=True, exist_ok=True)

class_dirs = sorted(
    [p for p in ASL_SRC.iterdir() if p.is_dir() and not p.name.startswith(".")],
    key=lambda p: p.name.lower(),
)

log: list[dict] = []

for cls in class_dirs:
    for vid in list_numeric_mp4s(cls):
        rel_class = vid.parent.name
        dst = ASL_DST / rel_class / vid.name
        log.append(normalize_clip_to_target_frames(vid, dst, TARGET_FRAMES))

print(f"Normalized {len(log)} clips → {ASL_DST.relative_to(PROJECT_ROOT)}")
print("Each output has", TARGET_FRAMES, "frames; duration preserved (fps scaled).")

_fi = sorted(r["n_in"] for r in log)
print("Original frame counts — min/med/max:", _fi[0], _fi[len(_fi) // 2], _fi[-1])

Normalized 180 clips → Videos/ASL_60
Each output has 60 frames; duration preserved (fps scaled).
Original frame counts — min/med/max: 27 65 255


## Optional: verify outputs are exactly **60 frames** each

In [5]:
def count_frames_by_decode(path: Path) -> int:
    cap = cv2.VideoCapture(str(path))
    if not cap.isOpened():
        return -1
    n = 0
    while True:
        ok, _ = cap.read()
        if not ok:
            break
        n += 1
    cap.release()
    return n


out_paths = sorted(ASL_DST.rglob("*.mp4"))
print("Total output mp4:", len(out_paths))

SPOT = min(48, len(out_paths))
bad: list[Path] = []
for i, p in enumerate(out_paths[:SPOT]):
    nc = count_frames_by_decode(p)
    cap = cv2.VideoCapture(str(p))
    w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    cap.release()
    if nc != TARGET_FRAMES:
        bad.append(p)
    tag = "" if nc == TARGET_FRAMES else "  ** !=60 **"
    if i < 15:
        print(nc, "frames", f"{w}x{h}", p.relative_to(PROJECT_ROOT), tag)

print(f"Decoded first {SPOT} files.")
if bad:
    print("Mismatches (!= TARGET_FRAMES):", len(bad))
    print(bad[:12])
else:
    print("Spot-check: each decoded file above has", TARGET_FRAMES, "frames.")

_meta_counts = []
for p in out_paths:
    cap = cv2.VideoCapture(str(p))
    _meta_counts.append(int(cap.get(cv2.CAP_PROP_FRAME_COUNT)))
    cap.release()
print("CAP_PROP_FRAME_COUNT unique values (metadata, may lie):", sorted(set(_meta_counts)))

Total output mp4: 180
60 frames 480x320 Videos/ASL_60/africa/0.mp4 
60 frames 320x240 Videos/ASL_60/africa/1.mp4 
60 frames 656x370 Videos/ASL_60/africa/10.mp4 
60 frames 480x320 Videos/ASL_60/africa/11.mp4 
60 frames 320x240 Videos/ASL_60/africa/12.mp4 
60 frames 1280x720 Videos/ASL_60/africa/13.mp4 
60 frames 1920x1080 Videos/ASL_60/africa/14.mp4 
60 frames 1920x1080 Videos/ASL_60/africa/15.mp4 
60 frames 1920x1080 Videos/ASL_60/africa/16.mp4 
60 frames 640x480 Videos/ASL_60/africa/17.mp4 
60 frames 1920x1080 Videos/ASL_60/africa/18.mp4 
60 frames 1920x1080 Videos/ASL_60/africa/19.mp4 
60 frames 1280x720 Videos/ASL_60/africa/2.mp4 
60 frames 1920x1080 Videos/ASL_60/africa/20.mp4 
60 frames 656x370 Videos/ASL_60/africa/21.mp4 
Decoded first 48 files.
Spot-check: each decoded file above has 60 frames.
CAP_PROP_FRAME_COUNT unique values (metadata, may lie): [60]


## Frame counts (**ASL_60** vs **SGSL**)

Decode-counts each `*.mp4` under **`Videos/ASL_60`** and **`Videos/SGSL`** (subfolders). Expect **ASL_60** clips to match **`TARGET_FRAMES`** after normalization; **SGSL** is reported for comparison.

(Run after the imports / path cell so **`PROJECT_ROOT`**, **`ASL_DST`**, **`SGSL_ROOT`**, and **`TARGET_FRAMES`** exist.)

In [6]:
if "ASL_DST" not in globals():
    ASL_DST = PROJECT_ROOT / "Videos" / "ASL_60"
if "SGSL_ROOT" not in globals():
    SGSL_ROOT = PROJECT_ROOT / "Videos" / "SGSL"

def _decode_frame_count(path: Path) -> int:
    cap = cv2.VideoCapture(str(path))
    if not cap.isOpened():
        return -1
    n = 0
    while True:
        ok, _ = cap.read()
        if not ok:
            break
        n += 1
    cap.release()
    return n


def _collect_mp4_under(root: Path) -> list[Path]:
    if not root.is_dir():
        return []
    # One level: root/<class>/<n>.mp4 (skip hidden dirs)
    out: list[Path] = []
    for cls in sorted(
        [x for x in root.iterdir() if x.is_dir() and not x.name.startswith(".")],
        key=lambda x: x.name.lower(),
    ):
        mp4s = sorted(cls.glob("*.mp4"), key=lambda f: int(f.stem))
        out.extend(mp4s)
    return out


def _summarize_dataset(root: Path) -> tuple[list[int], list[dict]]:
    paths = _collect_mp4_under(root)
    rows: list[dict] = []
    counts: list[int] = []
    for vid in paths:
        n = _decode_frame_count(vid)
        counts.append(n)
        cls_name = vid.parent.name
        try:
            rel = str(vid.relative_to(PROJECT_ROOT))
        except ValueError:
            rel = str(vid)
        rows.append({"path": rel, "class": cls_name, "stem": int(vid.stem), "frames": n})
    return counts, rows


_all_rows: list[dict] = []

for dataset_label, root in (("ASL_60", ASL_DST), ("SGSL", SGSL_ROOT)):
    counts, rows = _summarize_dataset(root)
    for row in rows:
        _all_rows.append({**row, "dataset": dataset_label})

    print(f"=== {dataset_label} === ({root.relative_to(PROJECT_ROOT)})")
    if not counts:
        print("  No videos found.")
        continue
    good = [c for c in counts if c >= 0]
    sg = sorted(good)
    med = sg[len(sg) // 2]
    print(f"  Files: {len(counts)}")
    print(
        "  Frame counts (decode): min — median — max:",
        min(good),
        "—",
        med,
        "—",
        max(good),
    )
    print("  Distinct lengths:", sorted(set(good)))
    bad_open = sum(1 for c in counts if c < 0)
    if bad_open:
        print(f"  Could not decode: {bad_open} file(s)")
    if dataset_label == "ASL_60":
        neq = [r for r in rows if r["frames"] != TARGET_FRAMES and r["frames"] >= 0]
        print(f"  Videos where frames != {TARGET_FRAMES}: {len(neq)}")
        for r in neq[:15]:
            print("   ", r["path"], r["frames"])
        if len(neq) > 15:
            print("    ... truncating ...")

try:
    import pandas as pd

    if _all_rows:
        _df = pd.DataFrame(_all_rows).sort_values(["dataset", "class", "stem"]).reset_index(drop=True)
        display(_df.groupby("dataset")["frames"].describe())
except ImportError:
    pass


=== ASL_60 === (Videos/ASL_60)
  Files: 180
  Frame counts (decode): min — median — max: 60 — 60 — 60
  Distinct lengths: [60]
  Videos where frames != 60: 0
=== SGSL === (Videos/SGSL)
  Files: 180
  Frame counts (decode): min — median — max: 60 — 60 — 60
  Distinct lengths: [60]


,count,mean,std,min,25%,50%,75%,max
dataset,,,,,,,,
ASL_60,180.0,60.0,0.0,60.0,60.0,60.0,60.0,60.0
SGSL,180.0,60.0,0.0,60.0,60.0,60.0,60.0,60.0
